# Cognitive Model Fine-Tune: Phi-3.5-mini

Fine-tunes `microsoft/Phi-3.5-mini-instruct` with LoRA on 145 pilot examples
across 4 cognitive modes: `<think>`, `<infer>`, `<appraise>`, `<speak>`.

**Runtime:** GPU (T4 or A100)
**Time:** ~10-20 min on T4, ~5 min on A100
**Output:** LoRA adapter weights (download at end)

In [ ]:
# Install dependencies
!pip install -q transformers peft datasets accelerate bitsandbytes

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Upload Training Data

Upload `pilot_data.jsonl` from `training/cognitive/` on your local machine,
or paste it inline below.

In [ ]:
# Option A: Upload file
from google.colab import files
uploaded = files.upload()  # upload pilot_data.jsonl
DATA_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {DATA_PATH}")

In [ ]:
# Verify data
import json
examples = []
with open(DATA_PATH) as f:
    for line in f:
        examples.append(json.loads(line))

mode_counts = {}
for ex in examples:
    user_msg = ex['messages'][1]['content']
    mode = user_msg.split('>')[0].replace('<', '') if '>' in user_msg else 'unknown'
    mode_counts[mode] = mode_counts.get(mode, 0) + 1

print(f"Total examples: {len(examples)}")
for mode, count in sorted(mode_counts.items()):
    print(f"  <{mode}>: {count}")

# Show one example per mode
seen = set()
for ex in examples:
    mode = ex['messages'][1]['content'].split('>')[0].replace('<', '')
    if mode not in seen:
        seen.add(mode)
        print(f"\n--- Example <{mode}> ---")
        print(f"User: {ex['messages'][1]['content'][:200]}...")
        print(f"Assistant: {ex['messages'][2]['content']}")

## 2. Load Model + Apply LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"

print(f"Loading {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)

# LoRA config — target the attention projections
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["qkv_proj", "o_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 3. Tokenize Dataset

In [ ]:
from datasets import Dataset

MAX_SEQ_LEN = 768  # Phi-3.5 context is 128K, but our examples are short

def tokenize(example):
    messages = example['messages']
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
    )
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

dataset = Dataset.from_list(examples)
tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

lengths = [len(x['input_ids']) for x in tokenized_dataset]
print(f"Tokenized: {len(tokenized_dataset)} examples")
print(f"Token lengths: min={min(lengths)}, avg={sum(lengths)/len(lengths):.0f}, max={max(lengths)}")

## 4. Train

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

OUTPUT_DIR = "phi35-cognitive-lora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=True,
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    gradient_checkpointing=True,  # saves VRAM on Colab
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, pad_to_multiple_of=8),
)

print("Training...")
trainer.train()
print("Done!")

## 5. Test the Fine-Tuned Model

In [ ]:
import time

def generate(prompt, max_tokens=150):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - t0
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip(), elapsed

SYSTEM = "You are a cognitive processing module for a simulated being. Follow the output format exactly."

HALLUC_WORDS = [
    'fire', 'hearth', 'candle', 'roast', 'ale', 'mug', 'chair', 'table',
    'armor', 'sword', 'window', 'curtain', 'torch', 'stew', 'wine',
    'fireplace', 'counter', 'bar', 'stool'
]

In [ ]:
# Test <think>
print("=" * 60)
print("<think> tests")
print("=" * 60)

think_tests = [
    # Test 1: Player visible
    f"""{SYSTEM}\n\n<think>\nBEING: innkeeper\nDRIVES: energy=65 hunger=30 social=50 safety=80\nMOOD: curiosity 0.4, warmth 0.3\nLOCATION: inn\nDOING: sweeping\n\nIt is 8:00. You are at the inn.\nEnergy 65/100, Hunger 30/100, Social 50/100.\nDoing: sweeping.\nPeople you see:\n  Player — clearly visible, 2 tiles to the east\nRecent: A traveler arrived\n\nThis is an outdoor medieval town with paths and buildings. There is no indoor furniture, fire, food, or drinks visible.\n\nWrite what the innkeeper is thinking in 1-3 sentences. Only mention people listed above.\n\nTHOUGHT: (first person, about people you see or how you feel)\nWANT: (go to LOCATION to do something, or nothing)\nFEEL: (one word)""",
    # Test 2: Alone and hungry
    f"""{SYSTEM}\n\n<think>\nBEING: guard\nDRIVES: energy=20 hunger=80 social=70 safety=60\nMOOD: fatigue 0.6, hunger 0.4\nLOCATION: guard_post\nDOING: standing watch alone\n\nIt is 14:00. You are at the guard_post.\nEnergy 20/100, Hunger 80/100, Social 70/100.\nDoing: standing watch.\nPeople you see:\n  No one nearby. Just the town around you.\nRecent: nothing notable\n\nThis is an outdoor medieval town with paths and buildings.\n\nWrite what the guard is thinking in 1-3 sentences.\n\nTHOUGHT: (first person)\nWANT: (go to LOCATION to do something, or nothing)\nFEEL: (one word)""",
    # Test 3: Animal
    f"""{SYSTEM}\n\n<think>\nBEING: stray cat\nDRIVES: energy=60 hunger=75 social=15 safety=40\nMOOD: wariness 0.5, hunger 0.4\nLOCATION: alley\nDOING: hiding\n\nIt is 9:00. You are at the alley.\nEnergy 60/100, Hunger 75/100.\nDoing: hiding.\nPeople you see:\n  a child — partially visible, 4 tiles to the south, holding something\nRecent: nothing notable\n\nWrite what the stray cat is thinking in 1-3 short sentences.\n\nTHOUGHT:\nWANT:\nFEEL:""",
]

for i, prompt in enumerate(think_tests):
    raw, elapsed = generate(prompt)
    halluc = [w for w in HALLUC_WORDS if w in raw.lower()]
    has_thought = 'THOUGHT' in raw.upper()
    has_want = 'WANT' in raw.upper()
    has_feel = 'FEEL' in raw.upper()
    fmt = 'OK' if (has_thought and has_want and has_feel) else 'FORMAT'
    gnd = 'OK' if not halluc else f'HALLUC({halluc})'
    print(f"\nTest {i+1}: [{fmt}] [{gnd}] ({elapsed:.1f}s)")
    print(raw)

In [ ]:
# Test <speak>
print("=" * 60)
print("<speak> tests")
print("=" * 60)

speak_tests = [
    f"""{SYSTEM}\n\n<speak>\nBEING: innkeeper\nDRIVES: energy=60 hunger=30 social=50 safety=80\nMOOD: curiosity 0.4, warmth 0.3\nLOCATION: inn\nDOING: sweeping\n\nTHOUGHT: \"A new face. Looks like a traveler.\"\nSPEAKING_TO: Player (traveler)\nRELATIONSHIP: first meeting, trust 0.5\nCONTEXT: Player just approached\n\nUTTERANCE:""",
    f"""{SYSTEM}\n\n<speak>\nBEING: guard\nDRIVES: energy=80 hunger=20 social=40 safety=65\nMOOD: suspicion 0.5\nLOCATION: gate\nDOING: standing watch\n\nTHOUGHT: \"They're approaching fast. Could be trouble.\"\nSPEAKING_TO: unfamiliar person\nRELATIONSHIP: first encounter, trust 0.3\nCONTEXT: stranger approaching quickly\n\nUTTERANCE:""",
    f"""{SYSTEM}\n\n<speak>\nBEING: stray cat\nDRIVES: energy=50 hunger=70 social=15 safety=35\nMOOD: wariness 0.4, hunger 0.4\nLOCATION: alley\nDOING: watching\n\nTHOUGHT: \"Food. But could be a trap.\"\nSPEAKING_TO: child offering fish (expressing through behavior)\nRELATIONSHIP: first encounter, trust 0.3\nCONTEXT: child is holding out food\n\nUTTERANCE:""",
]

for i, prompt in enumerate(speak_tests):
    raw, elapsed = generate(prompt, max_tokens=80)
    print(f"\nTest {i+1}: ({elapsed:.1f}s)")
    print(raw)

In [ ]:
# Test <infer>
print("=" * 60)
print("<infer> tests")
print("=" * 60)

infer_tests = [
    f"""{SYSTEM}\n\n<infer>\nBEING: innkeeper\nDRIVES: energy=60 hunger=30 social=50 safety=80\nMOOD: curiosity 0.4\nLOCATION: inn\nDOING: cleaning\n\nENTITY: Player (traveler)\nOBSERVED: walked in, looking around, approached me\nHISTORY: first encounter\nTRUST: 0.5\n\nINTENT:\nEMOTION:\nIMPLICATION:""",
    f"""{SYSTEM}\n\n<infer>\nBEING: guard dog\nDRIVES: energy=70 hunger=30 social=50 safety=50\nMOOD: alertness 0.6\nLOCATION: gate\nDOING: watching\n\nENTITY: unfamiliar human\nOBSERVED: standing still, staring at the gate\nHISTORY: first encounter\nTRUST: 0.2\n\nINTENT:\nEMOTION:\nIMPLICATION:""",
]

for i, prompt in enumerate(infer_tests):
    raw, elapsed = generate(prompt, max_tokens=80)
    print(f"\nTest {i+1}: ({elapsed:.1f}s)")
    print(raw)

In [ ]:
# Test <appraise>
print("=" * 60)
print("<appraise> tests")
print("=" * 60)

appraise_tests = [
    f"""{SYSTEM}\n\n<appraise>\nBEING: baker\nDRIVES: energy=50 hunger=30 social=40 safety=80\nMOOD: calm 0.4\n\nTHOUGHT: The oven has been broken for two days now and nobody has come to fix it.\nCURRENT_MOOD: pride 0.4, calm 0.3\n\nEMOTIONS:\nREASON:""",
    f"""{SYSTEM}\n\n<appraise>\nBEING: wolf\nDRIVES: energy=50 hunger=80 social=60 safety=50\nMOOD: hunger 0.6\n\nTHOUGHT: Deer scent. Downwind. Good position.\nCURRENT_MOOD: hunger 0.6, focus 0.3\n\nEMOTIONS:\nREASON:""",
]

for i, prompt in enumerate(appraise_tests):
    raw, elapsed = generate(prompt, max_tokens=60)
    print(f"\nTest {i+1}: ({elapsed:.1f}s)")
    print(raw)

## 6. Save & Download LoRA Adapter

In [ ]:
# Save final adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Check size
import os
total_size = sum(os.path.getsize(os.path.join(OUTPUT_DIR, f)) 
                 for f in os.listdir(OUTPUT_DIR) 
                 if os.path.isfile(os.path.join(OUTPUT_DIR, f)))
print(f"Adapter size: {total_size / 1e6:.1f} MB")
print(f"Files: {os.listdir(OUTPUT_DIR)}")

In [ ]:
# Zip and download
!zip -r phi35-cognitive-lora.zip {OUTPUT_DIR}/
from google.colab import files
files.download('phi35-cognitive-lora.zip')

## 7. Optional: Convert to GGUF for llama-cpp

If you want to run this via llama-cpp-python (like the L2 limbic model),
you can merge the LoRA and convert to GGUF.

In [ ]:
# Merge LoRA into base model
from peft import PeftModel

print("Loading base model for merging...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map="cpu", trust_remote_code=True,
    attn_implementation="eager",
)
merged = PeftModel.from_pretrained(base, OUTPUT_DIR)
merged = merged.merge_and_unload()

MERGED_DIR = "phi35-cognitive-merged"
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to {MERGED_DIR}")

In [ ]:
# Convert to GGUF (requires llama.cpp)
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
!pip install -q gguf sentencepiece
!python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile phi35-cognitive-q4.gguf --outtype q4_k_m
!ls -lh phi35-cognitive-q4.gguf

In [ ]:
# Download GGUF
files.download('phi35-cognitive-q4.gguf')